In [22]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))            # project folder (for games.py)
sys.path.insert(0, os.path.abspath('../starter'))    # starter folder (for baseline.py)

import numpy as np
import baseline

sys.path.insert(0, os.path.abspath('../evaluation'))
import possession_metrics as pm

gt = pm.label_codes(pm.read_label_csv('../evaluation/gt/FAKE_possession_gt.csv'))

def score(labels):
    r = pm.evaluate(gt, pm.label_codes(labels))
    return {'macroF1': round(r['macro_f1'], 4),
            'spellF1': round(r['spell_f1_mean'], 4),
            'possessions': r['n_pred_spells']}


codes = baseline.build()   # 0 = home, 1 = away, -1 = no answer

teams: H = FAKE Barcelona, A = FAKE Athletic Club
reading BAR-ATH_tactical_atd.txt ...
  ball located on 88717 of 199319 frames (44.5%)



In [12]:
## im going ot read the file and go through each line
## reading the ball information and storing them in an array sized 
## length of frames and then filling them witn nan


ball_x = np.full(len(codes), np.nan)
ball_y = np.full(len(codes), np.nan)
ball_speed = np.full(len(codes), np.nan)

with open(raw_path) as handle:
    for line in handle:
        parts = line.split(":")
        subParts = parts[2].split(",")
        frame = int(parts[0]) - 1
        ball_x[frame] = float(subParts[0])
        ball_y[frame] = float(subParts[1])
        ball_speed[frame] = float(subParts[2])

In [13]:
## How often does the baseline have no answer in the first half?
countBar = 0
countAlt = 0
countIdk = 0
start = 11185

firstHalf = codes[start:90652]
for i in range(len(firstHalf)):
    if (firstHalf[i] == 0):
            countBar += 1
    elif (firstHalf[i] == 1):
            countAlt += 1
    elif (firstHalf[i] == -1):
           countIdk += 1

print(countBar, countAlt, countIdk)

       


21936 21898 35633


In [14]:
   ## Where is the first frame with no answer?
for i in range(len(firstHalf)):
       if(firstHalf[i] == -1):
              print (start + i)
              break

       

11245


In [26]:
## now I want to find the gaps of -1
## goes through a while loop not a for loop to remember where i has been 
## check if the current is -1 and then continues until its not -1 anymore and then 
## prints the start and end of the gap and the length of the gap.
## mark one before -1 and the gap to see if they match
## add the logic to determine the ball being out
## create an empty list of gaps and appead the start and end information there
## right before a throw in fill in the gaps with -1
## I could work on the momembers before a throw in

i = 0
gapCount = 0
oneBeforeStart = -1
oneAfterEnd = -1
gaps = []


while(i <len(firstHalf)):
    if(firstHalf[i] == -1):
        gapCount += 1
        gapStart = i
        oneBeforeStart = firstHalf[i - 1] 
        while (i < len(firstHalf) and firstHalf[i] == -1):
            i += 1
        gapEnd = i - 1
        oneAfterEnd = firstHalf[i]
        print("Gap from", start + gapStart, "to", start + gapEnd, "length", gapEnd - gapStart + 1)
        print("Gap count:", gapCount)
        gaps.append((gapStart,gapEnd))
        break
    else:
        i += 1


Gap from 11245 to 11304 length 60
Gap count: 1


In [16]:
##Organizing the raw data for ball (x,y, speed)
## I just did for one ball, next time do a loop to get all the balls
## have an array to collect that data for each frame
## this reads ony one line


raw_path = "../data/tactical/BAR-ATH_tactical_atd.txt"

with open(raw_path) as handle:
    first_line = handle.readline()
    parts = first_line.split(":")

print(first_line[:200])
print(parts[0], parts[2])



1:NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,NaN,NaN;NaN,Na
1 NaN,NaN,NaN



In [24]:
## so just label the gaps and then create a new loops that checks the corresponding frames and if the gap has the right conditions then just fill -1?
filled = firstHalf.copy()

for gapStart, gapEnd in gaps:
    frame = start + gapEnd + 1                 # the frame where the ball comes back
    before = firstHalf[gapStart - 1]
    after = firstHalf[gapEnd + 1]

    if (ball_y[frame] < 0.08 or ball_y[frame] > 0.92) and codes[frame - 1] == -1:
        continue                               # dead ball → leave the gap as -1

    if before == after:
        filled[gapStart : gapEnd + 1] = before





    

In [ ]:
## for hard gaps known info team before, team after
## gap length, where the ball was last seen, 
## where the ball reappears, how far the ball traveled
## player position, player speed


In [25]:
myCodes = codes.copy()
myCodes[start:90652] = filled            # put your filled first half into the full match

print("baseline:", score(baseline.smooth(codes, 125)))
print("mine:    ", score(baseline.smooth(myCodes, 125)))

baseline: {'macroF1': 0.5239, 'spellF1': 0.3757, 'possessions': 554}
mine:     {'macroF1': 0.5161, 'spellF1': 0.401, 'possessions': 459}
